# A very brief introduction to MCMC methods

Markov-chain Monte Carlo (MCMC) methods are widely used for the practice of fitting models to data. Given a data set and a model with several parameters, we often want to know what combination of parameters fits the data. Strictly speaking, MCMC is not an optimizer: it does not necessarily find the best combination of parameters. What it is really good at though, is to show what ranges of parameters are probable, and to obtain uncertainty estimates for all parameters. This short notebook uses the Python package [emcee](https://emcee.readthedocs.io/en/stable/) and is based on [this tutorial](https://prappleizer.github.io/Tutorials/MCMC/MCMC_Tutorial.html). 

## Generation of data

We generate a data set that is an amplitude as a function of time, chosen to consist of three sinusoids with different amplitudes and periods. We add some noise to the data as well. 

In [ ]:
!pip install emcee

import numpy as np
import emcee
import matplotlib.pyplot as plt

# Amplitudes, periods and a constant
a1, a2, a3 = 0.5, 2.0, -1.2
p1, p2, p3 = 0.054867, 0.123, 0.34508640
const = 10.0

# Generate data set without noise
t = np.linspace(0,1,1000)
data = a1*np.sin(2*np.pi*t/p1) + a2*np.sin(2*np.pi*t/p2) + a3*np.sin(2*np.pi*t/p3) + const

# Add noise
noise_level = 0.5
rng = np.random.default_rng(seed=1)
data = data + rng.uniform(low=-noise_level, high=noise_level, size=np.shape(data))

# Show data
plt.plot(t, data)
plt.show()

## The Model

In this simple example, we make a model of three sinusoids (similar to the data without noise). The function below returns the predicted amplitude based on the vector $\theta$ that contains the model parameters, i.e. all amplitudes and periods. 

In [ ]:
def model(theta,t=t):
    a1,a2,a3,p1,p2,p3,const = theta
    model = a1*np.sin(2*np.pi*t/p1) + a2*np.sin(2*np.pi*t/p2) + a3*np.sin(2*np.pi*t/p3) + const
    return model

## Likeliness function

Next, we need a function that determines how likely a set of parameters is. The closer the model predictions to the data, the more likely this set of parameters is. Usually we can do a simple $\chi^2$ test. This function needs the error in the data.

In [ ]:
def lnlike(theta, x, y, yerr):
    LnLike = -0.5*(y - model(theta,t=x))**2/yerr/yerr
    return np.sum(LnLike)

## Encoding prior knowledge

It is likely that we know at least some available range for our parameters. If one of the model parameters is temperature, it should at least be positive. But there may be more constraints to narrow the allowed temperature range. Such information about the parameters is encoded in the following function. Convention is that if a set of parameters is valid, it should return zero, otherwise -infinity. Here we impose some arbitrary ranges on the amplitudes, and enforcing periods to be positive and smaller than unity.

In [ ]:
def lnprior(theta):
    a1, a2, a3, p1, p2, p3, const = theta
    if np.abs(a1) < 10 and np.abs(a2) < 10 and np.abs(a3) < 10 and p1 > 0 and p2 > 0 and p3 > 0 and p1 < 1 and p2 < 1 and p3 < 1 and const > 0:
        return 0.0
    else:
        return -np.inf

## Probability of parameters

The final function combines the previous two and give the likelyness of the set of parameters, but if the parameters are invalid it returns -infinity. 

In [ ]:
def lnprob(theta, x, y, yerr):
    lp = lnprior(theta)
    if lp == -np.inf:
        return -np.inf
    return lp + lnlike(theta, x, y, yerr)

## Main function

The main routine running MCMC uses the emcee package to create a sampler. This sampler creates a number of 'walkers' that will walk through parameter space for a certain number of iterations. At each iteration, the lnprob function will be called to judge the quality of the fit. It is standard practice to first do a 'burn-in': let the sampler settle in a bit to erase any artefacts of the initial condition. 

In [ ]:
def main(p0, nwalkers, niter, ndim, lnprob, args):
    sampler = emcee.EnsembleSampler(nwalkers, ndim, lnprob, args=args)

    print("Running burn-in...")
    p0, _, _ = sampler.run_mcmc(p0, 100)
    sampler.reset()

    print("Running production...")
    pos, prob, state = sampler.run_mcmc(p0, niter)

    return sampler, pos, prob, state


## Running MCMC

Now we are ready to run the MCMC method on our model and data. We need to specify the error on the data, the number of walkers, the number of iterations, and the initial conditions for each walker. Running MCMC gives us access of course to the best fit model (with best lnprob). 

In [ ]:
# Error on the data
data_err = 0.05*np.mean(data)

# Extra arguments to lnprob function (apart from theta)
args = (t, data, data_err)

# Initial theta
initial = np.array([1.0, 1.0, 1.0, 0.5, 0.5,0.5,1])
ndim = len(initial)

# Number of walkers
nwalkers = 500
# Number of iterations
niter = 1024
# Starting theta for each walker
p0 = [np.array(initial) + 1e-3 * np.random.randn(ndim) for i in range(nwalkers)]

# Run MCMC
sampler, pos, prob, state = main(p0, nwalkers, niter, ndim, lnprob, args)

# Select best fit
samples = sampler.flatchain
theta_best = samples[np.argmax(sampler.flatlnprobability)]

# Plot data plus best fit
plt.plot(t, data)
plt.plot(t, model(theta_best, t=t))
plt.show()

## Corner plots

As mentioned before, the power of MCMC does not lie in finding the optimum set of parameters (a better theta may be found by using an optimizer), but in giving information about the impact of various parameters and their uncertainty. The standard way of visualizing this is through 'corner plots'. This shows slices through the probability distribution, showing correllations between various parameters, and error estimates for each model parameter. Which paramaters are well-constrained? What happens if you change the number of walkers and/or iterations?

In [ ]:
import corner

labels = ['a1','a2','a3','p1','p2','p3','const']
fig = corner.corner(samples,show_titles=True,labels=labels,plot_datapoints=True,quantiles=[0.16, 0.5, 0.84])